In [4]:
from pathlib import Path

import pandas as pd

# --------------------------------------------------
# Project configuration
# --------------------------------------------------

PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"

DIVIPOLA_PATH = RAW_DATA_DIR / "divipola.csv"

print("Project root:", PROJECT_ROOT)
print("Raw data directory:", RAW_DATA_DIR)
print("DIVIPOLA file exists:", DIVIPOLA_PATH.exists())

Project root: c:\Users\Monica\Desktop\ETL\ODS2
Raw data directory: c:\Users\Monica\Desktop\ETL\ODS2\data\raw
DIVIPOLA file exists: False


In [5]:
# --------------------------------------------------
# Load DIVIPOLA dataset
# --------------------------------------------------

divipola = pd.read_csv(
    DIVIPOLA_PATH,
    dtype="string",
    encoding="utf-8-sig",
)

print("Dataset loaded successfully.")
print("Dimensions:", divipola.shape)

print("\nColumn names:")
print(divipola.columns.tolist())

print("\nFirst 5 rows:")
display(divipola.head())

FileNotFoundError: [Errno 2] No such file or directory: 'c:\\Users\\Monica\\Desktop\\ETL\\ODS2\\data\\raw\\divipola.csv'

In [ ]:
# --------------------------------------------------
# Initial data profiling: DIVIPOLA
# --------------------------------------------------

print("Dataset dimensions:", divipola.shape)

print("\nData types:")
display(divipola.dtypes.to_frame(name="data_type"))

print("\nMissing values per column:")
display(
    divipola.isna()
    .sum()
    .to_frame(name="missing_values")
)

print(
    "\nComplete duplicate rows:",
    divipola.duplicated().sum()
)

print("\nRecords by territorial division type:")
display(
    divipola[
        "Tipo: Municipio / Isla / Área no municipalizada"
    ].value_counts(dropna=False).to_frame(name="records")
)

Dataset dimensions: (1122, 7)

Data types:


,data_type
Código Departamento,string
Nombre Departamento,string
Código Municipio,string
Nombre Municipio,string
Tipo: Municipio / Isla / Área no municipalizada,string
longitud,string
Latitud,string



Missing values per column:


,missing_values
Código Departamento,0
Nombre Departamento,0
Código Municipio,0
Nombre Municipio,0
Tipo: Municipio / Isla / Área no municipalizada,0
longitud,0
Latitud,0



Complete duplicate rows: 0

Records by territorial division type:


,records
Tipo: Municipio / Isla / Área no municipalizada,
Municipio,1103
Área no municipalizada,18
Isla,1


In [ ]:
# --------------------------------------------------
# Validate municipality code uniqueness
# --------------------------------------------------

municipality_code = "Código Municipio"

print(
    "Missing municipality codes:",
    divipola[municipality_code].isna().sum()
)

print(
    "Duplicate municipality codes:",
    divipola[municipality_code].duplicated().sum()
)

print(
    "Unique municipality codes:",
    divipola[municipality_code].nunique()
)

# Display duplicated codes if any exist
duplicated_codes = divipola[
    divipola[municipality_code].duplicated(keep=False)
].sort_values(municipality_code)

print("\nRecords with duplicated municipality codes:")
display(duplicated_codes)

Missing municipality codes: 0
Duplicate municipality codes: 0
Unique municipality codes: 1122

Records with duplicated municipality codes:


,Código Departamento,Nombre Departamento,Código Municipio,Nombre Municipio,Tipo: Municipio / Isla / Área no municipalizada,longitud,Latitud


In [ ]:
# --------------------------------------------------
# Inspect coordinate formats
# --------------------------------------------------

coordinate_columns = ["longitud", "Latitud"]

for column in coordinate_columns:
    print(f"\nColumn: {column}")
    print("Data type:", divipola[column].dtype)
    print("Unique values:", divipola[column].nunique())
    print("Sample values:")
    display(divipola[column].head(10).to_frame())

print("\nCoordinate values containing commas:")
display(
    divipola[coordinate_columns]
    .apply(lambda column: column.str.contains(",", regex=False).sum())
    .to_frame(name="values_with_comma")
)


Column: longitud
Data type: string
Unique values: 1121
Sample values:


,longitud
0,"-75,581775"
1,"-75,428739"
2,"-76,064304"
3,"-75,141346"
4,"-75,702188"
5,"-75,077501"
6,"-75,878828"
7,"-75,711389"
8,"-75,335116"
9,"-75,148355"



Column: Latitud
Data type: string
Unique values: 1122
Sample values:


,Latitud
0,"6,246631"
1,"5,789315"
2,"6,632282"
3,"6,376061"
4,"6,038708"
5,"6,909655"
6,"5,657194"
7,"6,109719"
8,"6,885175"
9,"7,074703"



Coordinate values containing commas:


,values_with_comma
longitud,1122
Latitud,1122


In [ ]:
# --------------------------------------------------
# Create a working copy and convert coordinates
# --------------------------------------------------

divipola_work = divipola.copy()

for column in ["longitud", "Latitud"]:
    divipola_work[column] = pd.to_numeric(
        divipola_work[column].str.replace(",", ".", regex=False),
        errors="coerce",
    )

print("Original data types:")
display(divipola[["longitud", "Latitud"]].dtypes.to_frame("data_type"))

print("\nConverted data types:")
display(
    divipola_work[["longitud", "Latitud"]]
    .dtypes
    .to_frame("data_type")
)

print("\nMissing coordinates after conversion:")
display(
    divipola_work[["longitud", "Latitud"]]
    .isna()
    .sum()
    .to_frame("missing_values")
)

print("\nCoordinate summary:")
display(divipola_work[["longitud", "Latitud"]].describe())

Original data types:


,data_type
longitud,string
Latitud,string



Converted data types:


,data_type
longitud,Float64
Latitud,Float64



Missing coordinates after conversion:


,missing_values
longitud,0
Latitud,0



Coordinate summary:


,longitud,Latitud
count,1122.0,1122.0
mean,-74.651854,5.614861
std,1.675067,2.646144
min,-81.707181,-4.19895
25%,-75.767018,4.262183
50%,-74.767318,5.529295
75%,-73.449635,7.025522
max,-66.963692,13.373185


In [ ]:
# --------------------------------------------------
# Validate geographic coordinate ranges
# --------------------------------------------------

valid_longitude = divipola_work["longitud"].between(-82, -66)
valid_latitude = divipola_work["Latitud"].between(-5, 14)

invalid_coordinates = divipola_work[
    ~(valid_longitude & valid_latitude)
]

print("Records outside the broad geographic bounds:", len(invalid_coordinates))

print("\nInvalid coordinate records:")
display(
    invalid_coordinates[
        [
            "Código Departamento",
            "Nombre Departamento",
            "Código Municipio",
            "Nombre Municipio",
            "longitud",
            "Latitud",
        ]
    ]
)

print("\nCoordinate validation summary:")
print("Longitudes within bounds:", valid_longitude.sum())
print("Latitudes within bounds:", valid_latitude.sum())
print("Records within both bounds:", (valid_longitude & valid_latitude).sum())

Records outside the broad geographic bounds: 0

Invalid coordinate records:


,Código Departamento,Nombre Departamento,Código Municipio,Nombre Municipio,longitud,Latitud



Coordinate validation summary:
Longitudes within bounds: 1122
Latitudes within bounds: 1122
Records within both bounds: 1122


In [ ]:
# --------------------------------------------------
# Inspect territorial division categories
# --------------------------------------------------

division_type_column = (
    "Tipo: Municipio / Isla / Área no municipalizada"
)

print("Territorial division types:")
display(
    divipola_work[division_type_column]
    .value_counts(dropna=False)
    .rename_axis("division_type")
    .reset_index(name="records")
)

print("\nRecords that are not classified as municipalities:")
display(
    divipola_work.loc[
        divipola_work[division_type_column] != "Municipio",
        [
            "Código Departamento",
            "Nombre Departamento",
            "Código Municipio",
            "Nombre Municipio",
            division_type_column,
            "longitud",
            "Latitud",
        ],
    ]
)

Territorial division types:


,division_type,records
0,Municipio,1103
1,Área no municipalizada,18
2,Isla,1



Records that are not classified as municipalities:


,Código Departamento,Nombre Departamento,Código Municipio,Nombre Municipio,Tipo: Municipio / Isla / Área no municipalizada,longitud,Latitud
1087,88,"ARCHIPIÉLAGO DE SAN ANDRÉS, PROVIDENCIA Y SANT...",88001,SAN ANDRÉS,Isla,-81.707181,12.578108
1090,91,AMAZONAS,91263,EL ENCANTO,Área no municipalizada,-73.207114,-1.74806
1091,91,AMAZONAS,91405,LA CHORRERA,Área no municipalizada,-72.791889,-1.442617
1092,91,AMAZONAS,91407,LA PEDRERA,Área no municipalizada,-69.585499,-1.320301
1093,91,AMAZONAS,91430,LA VICTORIA,Área no municipalizada,-71.223208,0.054936
1094,91,AMAZONAS,91460,MIRITÍ - PARANÁ,Área no municipalizada,-70.98893,-0.888833
1095,91,AMAZONAS,91530,PUERTO ALEGRÍA,Área no municipalizada,-74.014461,-1.005674
1096,91,AMAZONAS,91536,PUERTO ARICA,Área no municipalizada,-71.752186,-2.147039
1098,91,AMAZONAS,91669,PUERTO SANTANDER,Área no municipalizada,-72.384213,-0.621184
1099,91,AMAZONAS,91798,TARAPACÁ,Área no municipalizada,-69.741745,-2.890126


In [ ]:
# --------------------------------------------------
# Final structural check: DIVIPOLA working dataset
# --------------------------------------------------

print("Dataset dimensions:", divipola_work.shape)

print("\nColumn names:")
print(divipola_work.columns.tolist())

print("\nData types:")
display(divipola_work.dtypes.to_frame(name="data_type"))

print("\nFirst 5 records:")
display(divipola_work.head())

print("\nFinal integrity checks:")
print("Unique municipality codes:", divipola_work["Código Municipio"].nunique())
print("Duplicate municipality codes:", divipola_work["Código Municipio"].duplicated().sum())
print("Missing coordinates:", divipola_work[["longitud", "Latitud"]].isna().sum().sum())

Dataset dimensions: (1122, 7)

Column names:
['Código Departamento', 'Nombre Departamento', 'Código Municipio', 'Nombre Municipio', 'Tipo: Municipio / Isla / Área no municipalizada', 'longitud', 'Latitud']

Data types:


,data_type
Código Departamento,string
Nombre Departamento,string
Código Municipio,string
Nombre Municipio,string
Tipo: Municipio / Isla / Área no municipalizada,string
longitud,Float64
Latitud,Float64



First 5 records:


,Código Departamento,Nombre Departamento,Código Municipio,Nombre Municipio,Tipo: Municipio / Isla / Área no municipalizada,longitud,Latitud
0,05,ANTIOQUIA,05001,MEDELLÍN,Municipio,-75.581775,6.246631
1,05,ANTIOQUIA,05002,ABEJORRAL,Municipio,-75.428739,5.789315
2,05,ANTIOQUIA,05004,ABRIAQUÍ,Municipio,-76.064304,6.632282
3,05,ANTIOQUIA,05021,ALEJANDRÍA,Municipio,-75.141346,6.376061
4,05,ANTIOQUIA,05030,AMAGÁ,Municipio,-75.702188,6.038708



Final integrity checks:
Unique municipality codes: 1122
Duplicate municipality codes: 0
Missing coordinates: 0


In [ ]:
# --------------------------------------------------
# Inventory raw data files
# --------------------------------------------------

print("Raw data directory:", RAW_DATA_DIR)

raw_files = sorted(
    path for path in RAW_DATA_DIR.iterdir()
    if path.is_file()
)

print(f"\nNumber of files: {len(raw_files)}")

for path in raw_files:
    print(f"{path.name} ({path.stat().st_size:,} bytes)")

Raw data directory: c:\Users\Monica\Desktop\ETL\ODS2\data\raw

Number of files: 3
divipola.csv (82,076 bytes)
Evaluaciones_Agropecuarias_Municipales_–_EVA._2019_-_2025._Base_Agrícola_20260905.csv (33,525,510 bytes)
open-meteo-52.55N13.41E38m.csv (7,987 bytes)


In [ ]:
# --------------------------------------------------
# Validate municipality codes between EVA and DIVIPOLA
# --------------------------------------------------

EVA_MUNICIPALITY_CODE = "Código Dane municipio"
DIVIPOLA_MUNICIPALITY_CODE = "Código Municipio"

# Extract and clean codes without modifying the original datasets
eva_codes = (
    eva[EVA_MUNICIPALITY_CODE]
    .astype("string")
    .str.strip()
)

divipola_codes = (
    divipola_work[DIVIPOLA_MUNICIPALITY_CODE]
    .astype("string")
    .str.strip()
)

# Inspect original code lengths
print("EVA municipality code lengths:")
display(
    eva_codes.str.len()
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("length")
    .reset_index(name="records")
)

print("\nDIVIPOLA municipality code lengths:")
display(
    divipola_codes.str.len()
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("length")
    .reset_index(name="records")
)

# Standardize codes to five characters for comparison
eva_codes_normalized = eva_codes.str.zfill(5)
divipola_codes_normalized = divipola_codes.str.zfill(5)

eva_unique_codes = set(eva_codes_normalized.dropna().unique())
divipola_unique_codes = set(divipola_codes_normalized.dropna().unique())

# Identify unmatched codes
eva_codes_not_in_divipola = sorted(
    eva_unique_codes - divipola_unique_codes
)

divipola_codes_not_in_eva = sorted(
    divipola_unique_codes - eva_unique_codes
)

print("\nComparison summary:")
print("Distinct municipality codes in EVA:", len(eva_unique_codes))
print("Distinct municipality codes in DIVIPOLA:", len(divipola_unique_codes))
print("EVA codes not found in DIVIPOLA:", len(eva_codes_not_in_divipola))
print("DIVIPOLA codes not used in EVA:", len(divipola_codes_not_in_eva))

print("\nFirst 20 EVA codes not found in DIVIPOLA:")
print(eva_codes_not_in_divipola[:20])

EVA municipality code lengths:


,length,records
0,5,166732



DIVIPOLA municipality code lengths:


,length,records
0,5,1122



Comparison summary:
Distinct municipality codes in EVA: 1103
Distinct municipality codes in DIVIPOLA: 1122
EVA codes not found in DIVIPOLA: 0
DIVIPOLA codes not used in EVA: 19

First 20 EVA codes not found in DIVIPOLA:
[]


In [ ]:
# --------------------------------------------------
# Build the territorial reference for EVA municipalities
# --------------------------------------------------

# One row per municipality code in EVA
eva_municipalities = (
    eva[
        [
            "Código Dane departamento",
            "Departamento",
            "Código Dane municipio",
            "Municipio",
        ]
    ]
    .drop_duplicates()
    .copy()
)

# Join EVA municipalities with DIVIPOLA coordinates
municipality_reference = eva_municipalities.merge(
    divipola_work[
        [
            "Código Municipio",
            "longitud",
            "Latitud",
            "Tipo: Municipio / Isla / Área no municipalizada",
        ]
    ],
    left_on="Código Dane municipio",
    right_on="Código Municipio",
    how="left",
    validate="many_to_one",
    indicator=True,
)

print("Reference dimensions:", municipality_reference.shape)

print("\nMerge status:")
display(
    municipality_reference["_merge"]
    .value_counts()
    .rename_axis("merge_status")
    .reset_index(name="records")
)

print("\nMissing coordinates:")
display(
    municipality_reference[["longitud", "Latitud"]]
    .isna()
    .sum()
    .to_frame(name="missing_values")
)

print("\nFirst 10 municipalities:")
display(municipality_reference.head(10))

Reference dimensions: (1103, 9)

Merge status:


,merge_status,records
0,both,1103
1,left_only,0
2,right_only,0



Missing coordinates:


,missing_values
longitud,0
Latitud,0



First 10 municipalities:


,Código Dane departamento,Departamento,Código Dane municipio,Municipio,Código Municipio,longitud,Latitud,Tipo: Municipio / Isla / Área no municipalizada,_merge
0,05,Antioquia,05001,Medellín,05001,-75.581775,6.246631,Municipio,both
1,05,Antioquia,05002,Abejorral,05002,-75.428739,5.789315,Municipio,both
2,05,Antioquia,05004,Abriaquí,05004,-76.064304,6.632282,Municipio,both
3,05,Antioquia,05021,Alejandría,05021,-75.141346,6.376061,Municipio,both
4,05,Antioquia,05030,Amagá,05030,-75.702188,6.038708,Municipio,both
5,05,Antioquia,05031,Amalfi,05031,-75.077501,6.909655,Municipio,both
6,05,Antioquia,05034,Andes,05034,-75.878828,5.657194,Municipio,both
7,05,Antioquia,05036,Angelópolis,05036,-75.711389,6.109719,Municipio,both
8,05,Antioquia,05038,Angostura,05038,-75.335116,6.885175,Municipio,both
9,05,Antioquia,05040,Anorí,05040,-75.148355,7.074703,Municipio,both


In [ ]:
# --------------------------------------------------
# Inspect EVA temporal coverage
# --------------------------------------------------

print("Year coverage:")
display(
    eva["Año"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("year")
    .reset_index(name="records")
)

print("\nPeriod values:")
display(
    eva["Periodo"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("period")
    .reset_index(name="records")
)

print("\nYear-period combinations:")
display(
    eva[["Año", "Periodo"]]
    .drop_duplicates()
    .sort_values(["Año", "Periodo"])
    .reset_index(drop=True)
)

Year coverage:


,year,records
0,2019,20436
1,2020,21511
2,2021,23898
3,2022,24808
4,2023,24909
5,2024,25553
6,2025,25617



Period values:


,period,records
0,2019,8007
1,2019A,6753
2,2019B,5676
3,2020,8572
4,2020A,6923
5,2020B,6016
6,2021,9527
7,2021A,7654
8,2021B,6717
9,2022,9627



Year-period combinations:


,Año,Periodo
0,2019,2019
1,2019,2019A
2,2019,2019B
3,2020,2020
4,2020,2020A
5,2020,2020B
6,2021,2021
7,2021,2021A
8,2021,2021B
9,2022,2022


In [ ]:
# Count distinct municipalities and crops for each period type

period_summary = (
    eva.assign(
        period_type=eva["Periodo"].astype("string").str.extract(
            r"(\d{4})([AB]?)"
        )[1].replace("", "Annual_or_unsuffixed")
    )
    .groupby(["Año", "period_type"], dropna=False)
    .agg(
        records=("Código Dane municipio", "size"),
        municipalities=("Código Dane municipio", "nunique"),
        crops=("Código del cultivo", "nunique"),
    )
    .reset_index()
    .sort_values(["Año", "period_type"])
)

display(period_summary)

,Año,period_type,records,municipalities,crops
0,2019,A,6753,1079,73
1,2019,Annual_or_unsuffixed,8007,1075,99
2,2019,B,5676,1040,72
3,2020,A,6923,1085,73
4,2020,Annual_or_unsuffixed,8572,1077,102
5,2020,B,6016,1069,73
6,2021,A,7654,1092,70
7,2021,Annual_or_unsuffixed,9527,1075,112
8,2021,B,6717,1075,70
9,2022,A,7818,1093,71


In [ ]:
columns_to_inspect = [
    "Código Dane municipio",
    "Municipio",
    "Cultivo",
    "Código del cultivo",
    "Año",
    "Periodo",
    "Área sembrada",
    "Área cosechada",
    "Producción",
    "Rendimiento",
]

for period in ["2025", "2025A", "2025B"]:
    print(f"\n--- Periodo: {period} ---")

    sample = (
        eva.loc[
            (eva["Año"] == 2025) &
            (eva["Periodo"] == period),
            columns_to_inspect,
        ]
        .head(8)
    )

    display(sample)


--- Periodo: 2025 ---


,Código Dane municipio,Municipio,Cultivo,Código del cultivo,Año,Periodo,Área sembrada,Área cosechada,Producción,Rendimiento
20,05001,Medellín,Brevo,2041100,2025,2025,"7,00","7,00","56,00",8.00
35,05001,Medellín,Café,2030300,2025,2025,"354,55","321,54","283,38",0.88
42,05001,Medellín,Caña,2030402,2025,2025,"62,00","62,00","3.100,00",50.00
106,05001,Medellín,Fresa,2042700,2025,2025,"24,00","22,00","264,00",12.00
127,05001,Medellín,Guayaba,2043000,2025,2025,"1,00","1,00","4,00",4.00
156,05001,Medellín,Limón,2043603,2025,2025,"6,82","3,02","24,16",8.00
163,05001,Medellín,Mandarina,2044101,2025,2025,"6,00","6,00","36,00",6.00
170,05001,Medellín,Mango,2044299,2025,2025,"4,00","4,00","32,00",8.00



--- Periodo: 2025A ---


,Código Dane municipio,Municipio,Cultivo,Código del cultivo,Año,Periodo,Área sembrada,Área cosechada,Producción,Rendimiento
8,05001,Medellín,Apio,1050600,2025,2025A,"5,50","5,50","104,50",19.00
13,05001,Medellín,Arveja,1060100,2025,2025A,"0,50","0,50","0,44",0.88
27,05001,Medellín,Brócoli,1050900,2025,2025A,"7,00","6,70","107,20",16.00
58,05001,Medellín,Cebolla de rama,1051200,2025,2025A,"70,00","67,00","1.206,00",18.00
72,05001,Medellín,Cilantro,1051300,2025,2025A,"9,50","9,50","66,50",7.00
92,05001,Medellín,Coliflor,1051500,2025,2025A,"2,50","2,00","40,00",20.00
119,05001,Medellín,Frijol,1060200,2025,2025A,"9,50","8,50","11,05",1.30
148,05001,Medellín,Lechuga,1051900,2025,2025A,"6,00","6,00","96,00",16.00



--- Periodo: 2025B ---


,Código Dane municipio,Municipio,Cultivo,Código del cultivo,Año,Periodo,Área sembrada,Área cosechada,Producción,Rendimiento
9,05001,Medellín,Apio,1050600,2025,2025B,"7,00","7,00","133,00",19.0
28,05001,Medellín,Brócoli,1050900,2025,2025B,"8,00","7,40","118,40",16.0
59,05001,Medellín,Cebolla de rama,1051200,2025,2025B,"71,00","70,00","1.260,00",18.0
73,05001,Medellín,Cilantro,1051300,2025,2025B,"10,00","9,00","63,00",7.0
93,05001,Medellín,Coliflor,1051500,2025,2025B,"3,00","2,50","50,00",20.0
120,05001,Medellín,Frijol,1060200,2025,2025B,"12,00","11,00","14,30",1.3
149,05001,Medellín,Lechuga,1051900,2025,2025B,"6,50","6,00","90,00",15.0
192,05001,Medellín,Otras hortalizas,1059900,2025,2025B,"27,00","25,00","175,00",7.0


In [ ]:
for period in ["2025", "2025A", "2025B"]:
    print(f"\n--- Periodo: {period} ---")

    subset = eva.loc[
        (eva["Año"] == 2025) &
        (eva["Periodo"] == period)
    ]

    for column in ["Grupo cultivo", "Ciclo del cultivo"]:
        print(f"\nDistribución de {column}:")

        display(
            subset[column]
            .value_counts(dropna=False)
            .rename_axis(column)
            .reset_index(name="records")
        )


--- Periodo: 2025 ---

Distribución de Grupo cultivo:


,Grupo cultivo,records
0,Frutales,7586
1,Cultivos tropicales tradicionales,1981
2,Oleaginosas,187
3,"Cultivos para condimentos, bebidas medicinales...",162
4,Raíces y tubérculos,6
5,Leguminosas,5
6,Hortalizas,2



Distribución de Ciclo del cultivo:


,Ciclo del cultivo,records
0,Permanente,9879
1,Transitorio,50



--- Periodo: 2025A ---

Distribución de Grupo cultivo:


...

,Ciclo del cultivo,records
0,Transitorio,8071



--- Periodo: 2025B ---

Distribución de Grupo cultivo:


,Grupo cultivo,records
0,Hortalizas,2413
1,Cereales,2216
2,Raíces y tubérculos,1522
3,Leguminosas,1311
4,Oleaginosas,68
5,Cultivos tropicales tradicionales,52
6,"Cultivos para condimentos, bebidas medicinales...",35



Distribución de Ciclo del cultivo:


,Ciclo del cultivo,records
0,Transitorio,7617


In [ ]:
# Define the date ranges corresponding to EVA reference periods

period_rows = []

for year in range(2019, 2026):
    period_rows.extend([
        {
            "Periodo": str(year),
            "period_type": "Annual",
            "start_date": f"{year}-01-01",
            "end_date": f"{year}-12-31",
        },
        {
            "Periodo": f"{year}A",
            "period_type": "Semester_A",
            "start_date": f"{year}-01-01",
            "end_date": f"{year}-06-30",
        },
        {
            "Periodo": f"{year}B",
            "period_type": "Semester_B",
            "start_date": f"{year}-07-01",
            "end_date": f"{year}-12-31",
        },
    ])

period_calendar = pd.DataFrame(period_rows)

period_calendar["start_date"] = pd.to_datetime(
    period_calendar["start_date"]
)
period_calendar["end_date"] = pd.to_datetime(
    period_calendar["end_date"]
)

display(period_calendar)

,Periodo,period_type,start_date,end_date
0,2019,Annual,2019-01-01,2019-12-31
1,2019A,Semester_A,2019-01-01,2019-06-30
2,2019B,Semester_B,2019-07-01,2019-12-31
3,2020,Annual,2020-01-01,2020-12-31
4,2020A,Semester_A,2020-01-01,2020-06-30
5,2020B,Semester_B,2020-07-01,2020-12-31
6,2021,Annual,2021-01-01,2021-12-31
7,2021A,Semester_A,2021-01-01,2021-06-30
8,2021B,Semester_B,2021-07-01,2021-12-31
9,2022,Annual,2022-01-01,2022-12-31


In [ ]:
# Validate that every EVA period has a corresponding date range

eva_periods = set(
    eva["Periodo"].astype("string").dropna().unique()
)

calendar_periods = set(
    period_calendar["Periodo"].astype("string").unique()
)

print("Distinct periods in EVA:", len(eva_periods))
print("Periods in calendar:", len(calendar_periods))

print("\nEVA periods missing from calendar:")
print(sorted(eva_periods - calendar_periods))

print("\nCalendar periods not found in EVA:")
print(sorted(calendar_periods - eva_periods))

print("\nDuplicate period keys in calendar:")
print(
    period_calendar.loc[
        period_calendar["Periodo"].duplicated(keep=False),
        "Periodo",
    ].tolist()
)

Distinct periods in EVA: 21
Periods in calendar: 21

EVA periods missing from calendar:
[]

Calendar periods not found in EVA:
[]

Duplicate period keys in calendar:
[]


In [ ]:
import requests

# Select three municipalities for a small API test
test_locations = (
    municipality_reference[
        [
            "Código Dane municipio",
            "Municipio",
            "Latitud",
            "longitud",
        ]
    ]
    .drop_duplicates(subset=["Código Dane municipio"])
    .head(3)
    .copy()
)

params = {
    "latitude": ",".join(test_locations["Latitud"].astype(str)),
    "longitude": ",".join(test_locations["longitud"].astype(str)),
    "start_date": "2019-01-01",
    "end_date": "2019-01-07",
    "daily": (
        "temperature_2m_mean,"
        "temperature_2m_max,"
        "temperature_2m_min,"
        "precipitation_sum,"
        "relative_humidity_2m_mean,"
        "et0_fao_evapotranspiration"
    ),
    "timezone": "America/Bogota",
    "format": "csv",
}

url = "https://archive-api.open-meteo.com/v1/archive"

response = requests.get(url, params=params, timeout=60)

print("HTTP status:", response.status_code)
print("Requested locations:")
display(test_locations)

print("\nFirst lines of API response:")
print("\n".join(response.text.splitlines()[:15]))

HTTP status: 200
Requested locations:


,Código Dane municipio,Municipio,Latitud,longitud
0,05001,Medellín,6.246631,-75.581775
1,05002,Abejorral,5.789315,-75.428739
2,05004,Abriaquí,6.632282,-76.064304



First lines of API response:
location_id,latitude,longitude,elevation,utc_offset_seconds,timezone,timezone_abbreviation
0,6.2214413,-75.55185,1475.0,-18000,America/Bogota,GMT-5
1,5.799649,-75.39935,2160.0,-18000,America/Bogota,GMT-5
2,6.7135324,-75.99664,1915.0,-18000,America/Bogota,GMT-5

location_id,time,temperature_2m_mean (°C),temperature_2m_max (°C),temperature_2m_min (°C),precipitation_sum (mm),relative_humidity_2m_mean (%),et0_fao_evapotranspiration (mm)
0,2019-01-01,20.0,27.7,13.9,0.00,77,4.71
0,2019-01-02,20.6,27.5,15.6,0.00,74,4.86
0,2019-01-03,19.6,26.8,13.5,0.00,76,4.44
0,2019-01-04,20.1,27.1,14.7,0.20,78,4.34
0,2019-01-05,21.3,29.1,16.3,0.50,79,4.06
0,2019-01-06,20.0,26.3,16.8,14.00,92,2.75
0,2019-01-07,18.9,22.9,15.5,1.90,91,2.14
1,2019-01-01,16.5,22.6,11.5,0.00,72,4.34
1,2019-01-02,17.0,22.2,13.4,0.00,70,4.46


In [ ]:
from io import StringIO

# Split the API response into metadata and daily observations
csv_lines = response.text.strip().splitlines()

daily_header_idx = next(
    i for i, line in enumerate(csv_lines)
    if line.startswith("location_id,time,")
)

metadata_lines = csv_lines[:daily_header_idx - 1]
daily_lines = csv_lines[daily_header_idx:]

location_metadata = pd.read_csv(
    StringIO("\n".join(metadata_lines))
)

weather_daily_test = pd.read_csv(
    StringIO("\n".join(daily_lines))
)

# Associate each API location ID with its requested municipality
location_map = test_locations.reset_index(drop=True).copy()
location_map["location_id"] = range(len(location_map))

weather_daily_test["location_id"] = (
    weather_daily_test["location_id"].astype(int)
)

weather_daily_test = weather_daily_test.merge(
    location_map[
        [
            "location_id",
            "Código Dane municipio",
            "Municipio",
        ]
    ],
    on="location_id",
    how="left",
    validate="many_to_one",
)

weather_daily_test["time"] = pd.to_datetime(
    weather_daily_test["time"]
)

print("Metadata dimensions:", location_metadata.shape)
print("Daily weather dimensions:", weather_daily_test.shape)

print("\nRows per municipality:")
display(
    weather_daily_test.groupby(
        ["Código Dane municipio", "Municipio"]
    ).size().reset_index(name="daily_records")
)

print("\nParsed daily weather data:")
display(weather_daily_test.head(10))

Metadata dimensions: (3, 7)
Daily weather dimensions: (21, 10)

Rows per municipality:


,Código Dane municipio,Municipio,daily_records
0,05001,Medellín,7
1,05002,Abejorral,7
2,05004,Abriaquí,7



Parsed daily weather data:


,location_id,time,temperature_2m_mean (°C),temperature_2m_max (°C),temperature_2m_min (°C),precipitation_sum (mm),relative_humidity_2m_mean (%),et0_fao_evapotranspiration (mm),Código Dane municipio,Municipio
0,0,2019-01-01,20.0,27.7,13.9,0.0,77,4.71,05001,Medellín
1,0,2019-01-02,20.6,27.5,15.6,0.0,74,4.86,05001,Medellín
2,0,2019-01-03,19.6,26.8,13.5,0.0,76,4.44,05001,Medellín
3,0,2019-01-04,20.1,27.1,14.7,0.2,78,4.34,05001,Medellín
4,0,2019-01-05,21.3,29.1,16.3,0.5,79,4.06,05001,Medellín
5,0,2019-01-06,20.0,26.3,16.8,14.0,92,2.75,05001,Medellín
6,0,2019-01-07,18.9,22.9,15.5,1.9,91,2.14,05001,Medellín
7,1,2019-01-01,16.5,22.6,11.5,0.0,72,4.34,05002,Abejorral
8,1,2019-01-02,17.0,22.2,13.4,0.0,70,4.46,05002,Abejorral
9,1,2019-01-03,16.2,21.3,11.7,0.0,72,4.12,05002,Abejorral


In [ ]:
from io import StringIO
import requests
import pandas as pd

# Request the complete study period for the three test municipalities
params_full_period = {
    "latitude": ",".join(test_locations["Latitud"].astype(str)),
    "longitude": ",".join(test_locations["longitud"].astype(str)),
    "start_date": "2019-01-01",
    "end_date": "2025-12-31",
    "daily": (
        "temperature_2m_mean,"
        "temperature_2m_max,"
        "temperature_2m_min,"
        "precipitation_sum,"
        "relative_humidity_2m_mean,"
        "et0_fao_evapotranspiration"
    ),
    "timezone": "America/Bogota",
    "format": "csv",
}

full_period_response = requests.get(
    "https://archive-api.open-meteo.com/v1/archive",
    params=params_full_period,
    timeout=120,
)

print("HTTP status:", full_period_response.status_code)
full_period_response.raise_for_status()

csv_lines = full_period_response.text.strip().splitlines()

daily_header_idx = next(
    i for i, line in enumerate(csv_lines)
    if line.startswith("location_id,time,")
)

weather_full_test = pd.read_csv(
    StringIO("\n".join(csv_lines[daily_header_idx:]))
)

weather_full_test["time"] = pd.to_datetime(
    weather_full_test["time"]
)

location_map = test_locations.reset_index(drop=True).copy()
location_map["location_id"] = range(len(location_map))

weather_full_test = weather_full_test.merge(
    location_map[
        ["location_id", "Código Dane municipio", "Municipio"]
    ],
    on="location_id",
    how="left",
    validate="many_to_one",
)

expected_days = len(
    pd.date_range("2019-01-01", "2025-12-31", freq="D")
)

print("Expected days per municipality:", expected_days)
print("Expected total records:", expected_days * len(test_locations))
print("Actual total records:", len(weather_full_test))
print("Missing municipality assignments:",
      weather_full_test["Código Dane municipio"].isna().sum())
print("Duplicate municipality-date keys:",
      weather_full_test.duplicated(
          ["Código Dane municipio", "time"]
      ).sum())

display(
    weather_full_test.groupby(
        ["Código Dane municipio", "Municipio"]
    ).agg(
        records=("time", "size"),
        first_date=("time", "min"),
        last_date=("time", "max"),
        distinct_dates=("time", "nunique"),
    ).reset_index()
)

HTTP status: 200
Expected days per municipality: 2557
Expected total records: 7671
Actual total records: 7671
Missing municipality assignments: 0
Duplicate municipality-date keys: 0


,Código Dane municipio,Municipio,records,first_date,last_date,distinct_dates
0,05001,Medellín,2557,2019-01-01,2025-12-31,2557
1,05002,Abejorral,2557,2019-01-01,2025-12-31,2557
2,05004,Abriaquí,2557,2019-01-01,2025-12-31,2557


In [ ]:
# Prepare municipality batches for the full weather extraction

municipality_locations = (
    municipality_reference[
        [
            "Código Dane municipio",
            "Municipio",
            "Latitud",
            "longitud",
        ]
    ]
    .drop_duplicates(subset=["Código Dane municipio"])
    .sort_values("Código Dane municipio")
    .reset_index(drop=True)
    .copy()
)

BATCH_SIZE = 25

municipality_locations["batch_id"] = (
    municipality_locations.index // BATCH_SIZE + 1
)

batch_plan = (
    municipality_locations
    .groupby("batch_id")
    .agg(
        municipalities=("Código Dane municipio", "size"),
        first_municipality=("Municipio", "first"),
        last_municipality=("Municipio", "last"),
    )
    .reset_index()
)

expected_daily_records = (
    len(municipality_locations) * expected_days
)

print("Municipalities to process:", len(municipality_locations))
print("Batch size:", BATCH_SIZE)
print("Number of batches:", len(batch_plan))
print("Expected daily records:", f"{expected_daily_records:,}")

print("\nMunicipalities per batch:")
display(batch_plan)

print("\nValidation:")
print(
    "Unique municipality codes:",
    municipality_locations["Código Dane municipio"].nunique()
)
print(
    "Missing coordinates:",
    municipality_locations[["Latitud", "longitud"]].isna().sum().sum()
)
print(
    "Municipalities without a batch:",
    municipality_locations["batch_id"].isna().sum()
)
print(
    "Maximum batch size:",
    batch_plan["municipalities"].max()
)

Municipalities to process: 1103
Batch size: 25
Number of batches: 45
Expected daily records: 2,820,371

Municipalities per batch:


,batch_id,municipalities,first_municipality,last_municipality
0,1,25,Medellín,Cáceres
1,2,25,Caicedo,Giraldo
2,3,25,Girardota,Nechí
3,4,25,Olaya,San Vicente Ferrer
4,5,25,Santa Bárbara,Zaragoza
5,6,25,Barranquilla,Achí
6,7,25,Altos del Rosario,Río Viejo
7,8,25,San Cristóbal,Berbeo
8,9,25,Betéitiva,Chivor
9,10,25,Duitama,Monguí



Validation:
Unique municipality codes: 1103
Missing coordinates: 0
Municipalities without a batch: 0
Maximum batch size: 25


In [ ]:
from pathlib import Path
import time
import requests
import pandas as pd
from io import StringIO

# ---------------------------------------------------------
# Output configuration
# ---------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent

WEATHER_OUTPUT_DIR = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "open_meteo_daily_batches"
)

WEATHER_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ---------------------------------------------------------
# Extraction configuration
# ---------------------------------------------------------

OPEN_METEO_URL = (
    "https://archive-api.open-meteo.com/v1/archive"
)

START_DATE = "2019-01-01"
END_DATE = "2025-12-31"

DAILY_VARIABLES = [
    "temperature_2m_mean",
    "temperature_2m_max",
    "temperature_2m_min",
    "precipitation_sum",
    "relative_humidity_2m_mean",
    "et0_fao_evapotranspiration",
]

TIMEZONE = "America/Bogota"

REQUEST_TIMEOUT = 180
MAX_RETRIES = 3
RETRY_DELAY_SECONDS = 5

# ---------------------------------------------------------
# Initial checks
# ---------------------------------------------------------

print("Output directory:", WEATHER_OUTPUT_DIR)
print("Output directory exists:", WEATHER_OUTPUT_DIR.exists())
print("Date range:", START_DATE, "to", END_DATE)
print("Daily variables:", len(DAILY_VARIABLES))
print("Number of batches:", len(batch_plan))
print("Maximum retries per batch:", MAX_RETRIES)

Output directory: c:\Users\Monica\Desktop\ETL\ODS2\data\raw\open_meteo_daily_batches
Output directory exists: True
Date range: 2019-01-01 to 2025-12-31
Daily variables: 6
Number of batches: 45
Maximum retries per batch: 3


In [ ]:
# ---------------------------------------------------------
# Extract and save batch 1
# ---------------------------------------------------------

batch_id = 1

batch_locations = (
    municipality_locations.loc[
        municipality_locations["batch_id"] == batch_id,
        [
            "Código Dane municipio",
            "Municipio",
            "Latitud",
            "longitud",
        ],
    ]
    .reset_index(drop=True)
    .copy()
)

params = {
    "latitude": ",".join(batch_locations["Latitud"].astype(str)),
    "longitude": ",".join(batch_locations["longitud"].astype(str)),
    "start_date": START_DATE,
    "end_date": END_DATE,
    "daily": ",".join(DAILY_VARIABLES),
    "timezone": TIMEZONE,
    "format": "csv",
}

response = None

for attempt in range(1, MAX_RETRIES + 1):
    try:
        print(f"Batch {batch_id}: request attempt {attempt}/{MAX_RETRIES}")

        candidate_response = requests.get(
            OPEN_METEO_URL,
            params=params,
            timeout=REQUEST_TIMEOUT,
        )

        candidate_response.raise_for_status()

        response = candidate_response
        break

    except requests.RequestException as exc:
        print(f"Request failed: {exc}")

        if attempt < MAX_RETRIES:
            time.sleep(RETRY_DELAY_SECONDS * attempt)

if response is None:
    raise RuntimeError(
        f"Batch {batch_id} failed after {MAX_RETRIES} attempts."
    )

# ---------------------------------------------------------
# Parse the CSV response
# ---------------------------------------------------------

csv_lines = response.text.strip().splitlines()

daily_header_idx = next(
    (
        i for i, line in enumerate(csv_lines)
        if line.startswith("location_id,time,")
    ),
    None,
)

if daily_header_idx is None:
    raise ValueError(
        "The response does not contain the expected daily data header. "
        f"Response preview:\n{response.text[:1000]}"
    )

metadata_lines = csv_lines[:daily_header_idx - 1]
daily_lines = csv_lines[daily_header_idx:]

location_metadata = pd.read_csv(
    StringIO("\n".join(metadata_lines))
)

weather_batch = pd.read_csv(
    StringIO("\n".join(daily_lines))
)

# ---------------------------------------------------------
# Map API location IDs to municipality codes
# ---------------------------------------------------------

location_map = batch_locations.copy()
location_map["location_id"] = range(len(location_map))

weather_batch = weather_batch.merge(
    location_map,
    on="location_id",
    how="left",
    validate="many_to_one",
)

# ---------------------------------------------------------
# Validate before saving
# ---------------------------------------------------------

weather_batch["time"] = pd.to_datetime(
    weather_batch["time"],
    errors="raise",
)

expected_batch_records = len(batch_locations) * expected_days

missing_municipality_codes = (
    weather_batch["Código Dane municipio"].isna().sum()
)

duplicate_keys = weather_batch.duplicated(
    subset=["Código Dane municipio", "time"]
).sum()

if len(weather_batch) != expected_batch_records:
    raise ValueError(
        f"Unexpected row count: {len(weather_batch):,}; "
        f"expected {expected_batch_records:,}."
    )

if missing_municipality_codes:
    raise ValueError(
        f"{missing_municipality_codes} rows could not be mapped to a municipality."
    )

if duplicate_keys:
    raise ValueError(
        f"Found {duplicate_keys} duplicate municipality-date keys."
    )

# ---------------------------------------------------------
# Save batch data and location metadata
# ---------------------------------------------------------

batch_file = WEATHER_OUTPUT_DIR / "weather_batch_001.csv"
metadata_file = WEATHER_OUTPUT_DIR / "location_metadata_batch_001.csv"

weather_batch.to_csv(
    batch_file,
    index=False,
)

location_metadata.to_csv(
    metadata_file,
    index=False,
)

print("\nBatch extraction completed successfully.")
print("Municipalities:", len(batch_locations))
print("Actual records:", f"{len(weather_batch):,}")
print("Expected records:", f"{expected_batch_records:,}")
print("Missing municipality assignments:", missing_municipality_codes)
print("Duplicate municipality-date keys:", duplicate_keys)
print("Date range:", weather_batch["time"].min(), "to", weather_batch["time"].max())
print("Data file:", batch_file)
print("Metadata file:", metadata_file)

display(weather_batch.head())

Batch 1: request attempt 1/3

Batch extraction completed successfully.
Municipalities: 25
Actual records: 63,925
Expected records: 63,925
Missing municipality assignments: 0
Duplicate municipality-date keys: 0
Date range: 2019-01-01 00:00:00 to 2025-12-31 00:00:00
Data file: c:\Users\Monica\Desktop\ETL\ODS2\data\raw\open_meteo_daily_batches\weather_batch_001.csv
Metadata file: c:\Users\Monica\Desktop\ETL\ODS2\data\raw\open_meteo_daily_batches\location_metadata_batch_001.csv


,location_id,time,temperature_2m_mean (°C),temperature_2m_max (°C),temperature_2m_min (°C),precipitation_sum (mm),relative_humidity_2m_mean (%),et0_fao_evapotranspiration (mm),Código Dane municipio,Municipio,Latitud,longitud
0,0,2019-01-01,20.0,27.7,13.9,0.0,77,4.71,05001,Medellín,6.246631,-75.581775
1,0,2019-01-02,20.6,27.5,15.6,0.0,74,4.86,05001,Medellín,6.246631,-75.581775
2,0,2019-01-03,19.6,26.8,13.5,0.0,76,4.44,05001,Medellín,6.246631,-75.581775
3,0,2019-01-04,20.1,27.1,14.7,0.2,78,4.34,05001,Medellín,6.246631,-75.581775
4,0,2019-01-05,21.3,29.1,16.3,0.5,79,4.06,05001,Medellín,6.246631,-75.581775


In [ ]:
# ---------------------------------------------------------
# Identify pending batches
# ---------------------------------------------------------

completed_batches = []

for batch_id in batch_plan["batch_id"]:
    batch_file = WEATHER_OUTPUT_DIR / f"weather_batch_{batch_id:03d}.csv"

    if batch_file.exists():
        try:
            existing_batch = pd.read_csv(
                batch_file,
                usecols=[
                    "Código Dane municipio",
                    "time",
                ],
            )

            expected_rows = (
                municipality_locations["batch_id"].eq(batch_id).sum()
                * expected_days
            )

            is_valid = (
                len(existing_batch) == expected_rows
                and existing_batch["Código Dane municipio"].notna().all()
                and not existing_batch.duplicated(
                    subset=["Código Dane municipio", "time"]
                ).any()
            )

            if is_valid:
                completed_batches.append(batch_id)

        except (ValueError, OSError, pd.errors.ParserError):
            pass

pending_batches = [
    batch_id
    for batch_id in batch_plan["batch_id"]
    if batch_id not in completed_batches
]

print("Total batches:", len(batch_plan))
print("Completed and validated batches:", len(completed_batches))
print("Pending batches:", len(pending_batches))
print("Pending batch IDs:", pending_batches)
print(
    "Expected records remaining:",
    f"{sum(municipality_locations['batch_id'].eq(b).sum() * expected_days for b in pending_batches):,}",
)

Total batches: 45
Completed and validated batches: 1
Pending batches: 44
Pending batch IDs: [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45]
Expected records remaining: 2,756,446


In [ ]:
# ---------------------------------------------------------
# Extract all pending weather batches
# ---------------------------------------------------------

batch_results = []
failed_batches = []

for batch_id in pending_batches:

    batch_file = WEATHER_OUTPUT_DIR / f"weather_batch_{batch_id:03d}.csv"
    metadata_file = (
        WEATHER_OUTPUT_DIR
        / f"location_metadata_batch_{batch_id:03d}.csv"
    )

    batch_locations = (
        municipality_locations.loc[
            municipality_locations["batch_id"] == batch_id,
            [
                "Código Dane municipio",
                "Municipio",
                "Latitud",
                "longitud",
            ],
        ]
        .reset_index(drop=True)
        .copy()
    )

    expected_batch_records = len(batch_locations) * expected_days

    print(
        f"\n{'=' * 60}\n"
        f"Batch {batch_id}/{len(batch_plan)} | "
        f"Municipalities: {len(batch_locations)} | "
        f"Expected rows: {expected_batch_records:,}"
    )

    params = {
        "latitude": ",".join(batch_locations["Latitud"].astype(str)),
        "longitude": ",".join(batch_locations["longitud"].astype(str)),
        "start_date": START_DATE,
        "end_date": END_DATE,
        "daily": ",".join(DAILY_VARIABLES),
        "timezone": TIMEZONE,
        "format": "csv",
    }

    try:
        response = None

        for attempt in range(1, MAX_RETRIES + 1):
            try:
                print(f"Request attempt {attempt}/{MAX_RETRIES}")

                candidate_response = requests.get(
                    OPEN_METEO_URL,
                    params=params,
                    timeout=REQUEST_TIMEOUT,
                )
                candidate_response.raise_for_status()

                response = candidate_response
                break

            except requests.RequestException as exc:
                print(f"Request failed: {exc}")

                if attempt < MAX_RETRIES:
                    time.sleep(RETRY_DELAY_SECONDS * attempt)

        if response is None:
            raise RuntimeError(
                f"Request failed after {MAX_RETRIES} attempts."
            )

        # Parse metadata and daily records from the API CSV response.
        csv_lines = response.text.strip().splitlines()

        daily_header_idx = next(
            (
                i for i, line in enumerate(csv_lines)
                if line.startswith("location_id,time,")
            ),
            None,
        )

        if daily_header_idx is None:
            raise ValueError(
                "The response does not contain the expected daily data header."
            )

        metadata_lines = csv_lines[:daily_header_idx - 1]
        daily_lines = csv_lines[daily_header_idx:]

        location_metadata = pd.read_csv(
            StringIO("\n".join(metadata_lines))
        )

        weather_batch = pd.read_csv(
            StringIO("\n".join(daily_lines))
        )

        # Map each API location_id to its input municipality.
        location_map = batch_locations.copy()
        location_map["location_id"] = range(len(location_map))

        weather_batch = weather_batch.merge(
            location_map,
            on="location_id",
            how="left",
            validate="many_to_one",
        )

        weather_batch["time"] = pd.to_datetime(
            weather_batch["time"],
            errors="raise",
        )

        # Validate before writing any output files.
        missing_codes = int(
            weather_batch["Código Dane municipio"].isna().sum()
        )

        duplicate_keys = int(
            weather_batch.duplicated(
                subset=["Código Dane municipio", "time"]
            ).sum()
        )

        actual_records = len(weather_batch)

        if actual_records != expected_batch_records:
            raise ValueError(
                f"Row count mismatch: {actual_records:,} received, "
                f"{expected_batch_records:,} expected."
            )

        if missing_codes:
            raise ValueError(
                f"{missing_codes} records lack a municipality code."
            )

        if duplicate_keys:
            raise ValueError(
                f"{duplicate_keys} duplicate municipality-date keys found."
            )

        # Save only after validation succeeds.
        weather_batch.to_csv(
            batch_file,
            index=False,
        )

        location_metadata.to_csv(
            metadata_file,
            index=False,
        )

        batch_results.append({
            "batch_id": batch_id,
            "municipalities": len(batch_locations),
            "records": actual_records,
            "status": "success",
        })

        print(f"Saved: {batch_file.name}")
        print(f"Validated records: {actual_records:,}")

    except Exception as exc:
        failed_batches.append({
            "batch_id": batch_id,
            "error": str(exc),
        })

        print(f"Batch {batch_id} FAILED: {exc}")

    # Brief pause between batches to reduce request pressure.
    time.sleep(2)

# ---------------------------------------------------------
# Extraction summary
# ---------------------------------------------------------

print("\n" + "=" * 60)
print("EXTRACTION SUMMARY")
print("=" * 60)

print("Batches attempted:", len(pending_batches))
print("Successful batches:", len(batch_results))
print("Failed batches:", len(failed_batches))
print(
    "Records saved in this run:",
    f"{sum(item['records'] for item in batch_results):,}",
)

if batch_results:
    display(pd.DataFrame(batch_results))

if failed_batches:
    print("\nFailed batches:")
    display(pd.DataFrame(failed_batches))

# ---------------------------------------------------------
# Diagnose the last Open-Meteo HTTP response
# ---------------------------------------------------------

print("Last response status:", response.status_code)

print("\nRelevant response headers:")

for header in [
    "Retry-After",
    "RateLimit-Limit",
    "RateLimit-Remaining",
    "RateLimit-Reset",
    "X-RateLimit-Limit",
    "X-RateLimit-Remaining",
    "X-RateLimit-Reset",
]:
    print(f"{header}: {response.headers.get(header, 'Not provided')}")

print("\nResponse body preview:")
print(response.text[:1500])

print("\nSaved batch files:")

saved_files = sorted(
    WEATHER_OUTPUT_DIR.glob("weather_batch_*.csv")
)

for file in saved_files:
    print(file.name)

print("\nNumber of saved batch files:", len(saved_files))


Batch 2/45 | Municipalities: 25 | Expected rows: 63,925
Request attempt 1/3
Saved: weather_batch_002.csv
Validated records: 63,925

Batch 3/45 | Municipalities: 25 | Expected rows: 63,925
Request attempt 1/3
Request failed: 429 Client Error: Too Many Requests for url: https://archive-api.open-meteo.com/v1/archive?latitude=6.379472%2C6.683269%2C6.142892%2C6.815069%2C6.27787%2C6.232461%2C6.206757%2C5.799461%2C6.175079%2C7.171629%2C5.597542%2C5.789748%2C6.028062%2C6.145238%2C5.743808%2C5.973845%2C6.677316%2C6.552116%2C6.173995%2C5.946313%2C6.97771%2C7.242875%2C5.610777%2C8.434526%2C8.094129&longitude=-75.444238%2C-75.220018%2C-75.184446%2C-75.239862%2C-75.441612%2C-75.160041%2C-75.734322%2C-75.906587%2C-75.612056%2C-75.764673%2C-75.818982%2C-75.785499%2C-75.429433%2C-75.637708%2C-75.60781%2C-75.360874%2C-75.812838%2C-74.78716%2C-75.339345%2C-75.523455%2C-76.817485%2C-76.435875%2C-75.176262%2C-76.787271%2C-74.77647&start_date=2019-01-01&end_date=2025-12-31&daily=temperature_2m_mean%2Ctemp

KeyboardInterrupt: 

In [ ]:
# ---------------------------------------------------------
# Reconcile saved weather batches
# ---------------------------------------------------------

saved_batch_ids = []
invalid_batch_ids = []

for batch_id in batch_plan["batch_id"]:
    batch_file = WEATHER_OUTPUT_DIR / f"weather_batch_{batch_id:03d}.csv"

    if not batch_file.exists():
        continue

    expected_rows = (
        municipality_locations["batch_id"].eq(batch_id).sum()
        * expected_days
    )

    try:
        existing_batch = pd.read_csv(
            batch_file,
            usecols=["Código Dane municipio", "time"],
        )

        is_valid = (
            len(existing_batch) == expected_rows
            and existing_batch["Código Dane municipio"].notna().all()
            and not existing_batch.duplicated(
                subset=["Código Dane municipio", "time"]
            ).any()
        )

        if is_valid:
            saved_batch_ids.append(batch_id)
        else:
            invalid_batch_ids.append(batch_id)

    except Exception as exc:
        invalid_batch_ids.append(batch_id)
        print(f"Batch {batch_id}: validation error: {exc}")

pending_batch_ids = [
    batch_id
    for batch_id in batch_plan["batch_id"]
    if batch_id not in saved_batch_ids
]

print("Validated batches:", saved_batch_ids)
print("Invalid batches:", invalid_batch_ids)
print("Pending batches:", len(pending_batch_ids))
print("Pending IDs:", pending_batch_ids)
print(
    "Validated records:",
    f"{sum(municipality_locations['batch_id'].eq(b).sum() * expected_days for b in saved_batch_ids):,}",
)

Validated batches: [1, 2, 3]
Invalid batches: []
Pending batches: 42
Pending IDs: [4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45]
Validated records: 191,775


In [ ]:
# ---------------------------------------------------------
# Conservative retry configuration
# ---------------------------------------------------------

MAX_RETRIES = 5

# Wait progressively longer after failures.
RETRY_BASE_DELAY_SECONDS = 60
RETRY_MAX_DELAY_SECONDS = 900

# Pause between successful batch requests.
BATCH_PAUSE_SECONDS = 30

# If HTTP 429 persists, wait before trying again.
RATE_LIMIT_COOLDOWN_SECONDS = 900

print("Pending batches:", len(pending_batch_ids))
print("Maximum retries:", MAX_RETRIES)
print("Initial retry delay:", RETRY_BASE_DELAY_SECONDS, "seconds")
print("Maximum retry delay:", RETRY_MAX_DELAY_SECONDS, "seconds")
print("Pause between batches:", BATCH_PAUSE_SECONDS, "seconds")
print("Rate-limit cooldown:", RATE_LIMIT_COOLDOWN_SECONDS, "seconds")
print("No API requests have been made by this cell.")

Pending batches: 42
Maximum retries: 5
Initial retry delay: 60 seconds
Maximum retry delay: 900 seconds
Pause between batches: 30 seconds
Rate-limit cooldown: 900 seconds
No API requests have been made by this cell.


In [ ]:
# ---------------------------------------------------------
# Resume extraction: controlled test for batch 4
# ---------------------------------------------------------

import time
import requests
import pandas as pd

from io import StringIO
from pathlib import Path

batch_id = 4

weather_file = WEATHER_OUTPUT_DIR / f"weather_batch_{batch_id:03d}.csv"
metadata_file = WEATHER_OUTPUT_DIR / f"location_metadata_batch_{batch_id:03d}.csv"

# Select the municipalities assigned to batch 4.
batch_locations = (
    municipality_locations
    .loc[municipality_locations["batch_id"] == batch_id]
    .copy()
    .reset_index(drop=True)
)

if batch_locations.empty:
    raise ValueError(f"No municipalities found for batch {batch_id}.")

if weather_file.exists() and metadata_file.exists():
    raise FileExistsError(
        "Batch 4 already has both output files. "
        "Validate them before attempting another request."
    )

print(f"Batch: {batch_id}")
print(f"Municipalities: {len(batch_locations)}")
print(f"Waiting {RATE_LIMIT_COOLDOWN_SECONDS} seconds before the test...")
print("No new API request will be made during this wait.")

time.sleep(RATE_LIMIT_COOLDOWN_SECONDS)

# One request only; no automatic retries in this diagnostic cell.
params = {
    "latitude": ",".join(batch_locations["Latitud"].astype(str)),
    "longitude": ",".join(batch_locations["longitud"].astype(str)),
    "start_date": START_DATE,
    "end_date": END_DATE,
    "daily": ",".join(DAILY_VARIABLES),
    "timezone": TIMEZONE,
    "format": "csv",
}

try:
    test_response = requests.get(
        OPEN_METEO_URL,
        params=params,
        timeout=REQUEST_TIMEOUT,
    )

    print("HTTP status:", test_response.status_code)
    print(
        "Retry-After:",
        test_response.headers.get("Retry-After", "Not provided")
    )

    if test_response.status_code == 429:
        print("Rate limit persists. Batch 4 was not saved.")
        print("Response details:", test_response.text[:1000])

    else:
        test_response.raise_for_status()

        csv_lines = test_response.text.strip().splitlines()

        daily_header_idx = next(
            (
                i for i, line in enumerate(csv_lines)
                if line.startswith("location_id,time,")
            ),
            None,
        )

        if daily_header_idx is None:
            raise ValueError(
                "Could not find the daily data header in the API response."
            )

        metadata_lines = csv_lines[:daily_header_idx - 1]
        daily_lines = csv_lines[daily_header_idx:]

        location_metadata = pd.read_csv(
            StringIO("\n".join(metadata_lines))
        )
        weather_batch = pd.read_csv(
            StringIO("\n".join(daily_lines))
        )

        # Map API location IDs to the original municipality records.
        location_map = batch_locations.copy()
        location_map["location_id"] = range(len(location_map))

        weather_batch = weather_batch.merge(
            location_map,
            on="location_id",
            how="left",
            validate="many_to_one",
        )

        weather_batch["time"] = pd.to_datetime(
            weather_batch["time"],
            errors="raise",
        )

        expected_rows = len(batch_locations) * expected_days

        missing_codes = weather_batch["Código Dane municipio"].isna().sum()
        duplicate_keys = weather_batch.duplicated(
            subset=["Código Dane municipio", "time"]
        ).sum()

        print("Rows received:", len(weather_batch))
        print("Expected rows:", expected_rows)
        print("Missing municipality codes:", missing_codes)
        print("Duplicate municipality-date keys:", duplicate_keys)

        if (
            len(weather_batch) != expected_rows
            or missing_codes > 0
            or duplicate_keys > 0
        ):
            raise ValueError(
                "Validation failed. Batch files will not be saved."
            )

        # Save only after the basic integrity checks pass.
        weather_batch.to_csv(weather_file, index=False)
        location_metadata.to_csv(metadata_file, index=False)

        print("Batch 4 saved successfully.")
        print("Weather file:", weather_file)
        print("Metadata file:", metadata_file)

except requests.RequestException as exc:
    print("Request failed:", str(exc))

    if "test_response" in locals():
        print("HTTP status:", test_response.status_code)
        print(
            "Retry-After:",
            test_response.headers.get("Retry-After", "Not provided")
        )
        print("Response details:", test_response.text[:1000])

    print("No further batches were attempted.")

Batch: 4
Municipalities: 25
Waiting 900 seconds before the test...
No new API request will be made during this wait.
HTTP status: 429
Retry-After: Not provided
Rate limit persists. Batch 4 was not saved.
Response details: {"error":true,"reason":"Daily API request limit exceeded. Please try again tomorrow."}


In [ ]:
# =========================================================
# RESUMABLE OPEN-METEO EXTRACTION
# Continues from existing validated batch files
# =========================================================

import time
import requests
import pandas as pd

from io import StringIO
from pathlib import Path


# ---------------------------------------------------------
# Configuration
# ---------------------------------------------------------

BATCH_PAUSE_SECONDS = 30
MAX_RETRIES = 3
RETRY_BASE_DELAY_SECONDS = 60

EXPECTED_DAYS = (
    pd.Timestamp(END_DATE) - pd.Timestamp(START_DATE)
).days + 1

WEATHER_OUTPUT_DIR = Path(WEATHER_OUTPUT_DIR)
WEATHER_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------
# Validate existing batch files
# ---------------------------------------------------------

def check_batch(batch_id, locations):
    weather_file = WEATHER_OUTPUT_DIR / (
        f"weather_batch_{batch_id:03d}.csv"
    )
    metadata_file = WEATHER_OUTPUT_DIR / (
        f"location_metadata_batch_{batch_id:03d}.csv"
    )

    if not weather_file.exists() or not metadata_file.exists():
        return False

    try:
        df = pd.read_csv(weather_file)

        expected_rows = len(locations) * EXPECTED_DAYS

        if len(df) != expected_rows:
            return False

        if df["Código Dane municipio"].isna().any():
            return False

        if df.duplicated(
            subset=["Código Dane municipio", "time"]
        ).any():
            return False

        df["time"] = pd.to_datetime(
            df["time"], errors="raise"
        )

        if (
            df["time"].min() != pd.Timestamp(START_DATE)
            or df["time"].max() != pd.Timestamp(END_DATE)
        ):
            return False

        expected_codes = set(
            locations["Código Dane municipio"].astype(str)
        )
        actual_codes = set(
            df["Código Dane municipio"].astype(str)
        )

        if expected_codes != actual_codes:
            return False

        counts = df.groupby(
            "Código Dane municipio"
        )["time"].nunique()

        if not (counts == EXPECTED_DAYS).all():
            return False

        return True

    except Exception as exc:
        print(f"Batch {batch_id}: validation error: {exc}")
        return False


# ---------------------------------------------------------
# Reconcile the batch plan with existing files
# ---------------------------------------------------------

batch_ids = sorted(
    municipality_locations["batch_id"].dropna().astype(int).unique()
)

validated_batches = []
pending_batches = []
invalid_existing_files = []

for batch_id in batch_ids:

    locations = municipality_locations.loc[
        municipality_locations["batch_id"] == batch_id
    ].copy()

    weather_file = WEATHER_OUTPUT_DIR / (
        f"weather_batch_{batch_id:03d}.csv"
    )
    metadata_file = WEATHER_OUTPUT_DIR / (
        f"location_metadata_batch_{batch_id:03d}.csv"
    )

    if check_batch(batch_id, locations):
        validated_batches.append(batch_id)
    else:
        pending_batches.append(batch_id)

        # Do not overwrite incomplete or invalid files automatically.
        if weather_file.exists() or metadata_file.exists():
            invalid_existing_files.append(batch_id)


print("Validated batches:", validated_batches)
print("Pending batches:", pending_batches)
print("Pending batch count:", len(pending_batches))

expected_pending_rows = sum(
    len(
        municipality_locations.loc[
            municipality_locations["batch_id"] == batch_id
        ]
    ) * EXPECTED_DAYS
    for batch_id in pending_batches
)

print("Expected pending rows:", f"{expected_pending_rows:,}")


if invalid_existing_files:
    raise RuntimeError(
        "Some pending batches already have incomplete or invalid files: "
        f"{invalid_existing_files}. "
        "Inspect these files before continuing; they will not be overwritten."
    )


# ---------------------------------------------------------
# Extract pending batches
# ---------------------------------------------------------

successful_this_run = 0
quota_exceeded = False

for position, batch_id in enumerate(pending_batches, start=1):

    batch_locations = (
        municipality_locations.loc[
            municipality_locations["batch_id"] == batch_id
        ]
        .copy()
        .reset_index(drop=True)
    )

    weather_file = WEATHER_OUTPUT_DIR / (
        f"weather_batch_{batch_id:03d}.csv"
    )
    metadata_file = WEATHER_OUTPUT_DIR / (
        f"location_metadata_batch_{batch_id:03d}.csv"
    )

    print(
        f"\n[{position}/{len(pending_batches)}] "
        f"Batch {batch_id}: "
        f"{len(batch_locations)} municipalities"
    )

    params = {
        "latitude": ",".join(
            batch_locations["Latitud"].astype(str)
        ),
        "longitude": ",".join(
            batch_locations["longitud"].astype(str)
        ),
        "start_date": START_DATE,
        "end_date": END_DATE,
        "daily": ",".join(DAILY_VARIABLES),
        "timezone": TIMEZONE,
        "format": "csv",
    }

    response = None

    for attempt in range(1, MAX_RETRIES + 1):

        try:
            candidate = requests.get(
                OPEN_METEO_URL,
                params=params,
                timeout=REQUEST_TIMEOUT,
            )

            if candidate.status_code == 429:
                body = candidate.text[:1500]
                print("HTTP 429:", body)

                if "daily api request limit exceeded" in body.lower():
                    print(
                        "Daily API quota exhausted. "
                        "Stopping without attempting further batches."
                    )
                    quota_exceeded = True
                    break

                if attempt < MAX_RETRIES:
                    delay = RETRY_BASE_DELAY_SECONDS * (2 ** (attempt - 1))
                    print(f"Rate limited; waiting {delay} seconds.")
                    time.sleep(delay)
                    continue

                candidate.raise_for_status()

            candidate.raise_for_status()
            response = candidate
            break

        except requests.RequestException as exc:
            print(f"Request error: {exc}")

            if attempt < MAX_RETRIES:
                delay = RETRY_BASE_DELAY_SECONDS * (2 ** (attempt - 1))
                print(f"Retrying in {delay} seconds.")
                time.sleep(delay)
            else:
                print("Retries exhausted. Stopping extraction.")
                break

    if quota_exceeded:
        break

    if response is None:
        print(f"Batch {batch_id} was not saved.")
        break

    # -----------------------------------------------------
    # Parse Open-Meteo CSV response
    # -----------------------------------------------------

    csv_lines = response.text.strip().splitlines()

    header_idx = next(
        (
            i for i, line in enumerate(csv_lines)
            if line.startswith("location_id,time,")
        ),
        None,
    )

    if header_idx is None or header_idx < 2:
        raise ValueError(
            f"Batch {batch_id}: daily CSV header not found."
        )

    metadata_lines = csv_lines[:header_idx - 1]
    daily_lines = csv_lines[header_idx:]

    metadata = pd.read_csv(
        StringIO("\n".join(metadata_lines))
    )
    weather = pd.read_csv(
        StringIO("\n".join(daily_lines))
    )

    # Map API location IDs to municipality records.
    location_map = batch_locations.copy()
    location_map["location_id"] = range(len(location_map))

    weather = weather.merge(
        location_map,
        on="location_id",
        how="left",
        validate="many_to_one",
    )

    weather["time"] = pd.to_datetime(
        weather["time"], errors="raise"
    )

    # -----------------------------------------------------
    # Quality validation before saving
    # -----------------------------------------------------

    expected_rows = len(batch_locations) * EXPECTED_DAYS

    missing_codes = weather[
        "Código Dane municipio"
    ].isna().sum()

    duplicate_keys = weather.duplicated(
        subset=["Código Dane municipio", "time"]
    ).sum()

    counts = weather.groupby(
        "Código Dane municipio"
    )["time"].nunique()

    complete_coverage = (
        len(counts) == len(batch_locations)
        and (counts == EXPECTED_DAYS).all()
    )

    valid = (
        len(weather) == expected_rows
        and missing_codes == 0
        and duplicate_keys == 0
        and complete_coverage
        and weather["time"].min() == pd.Timestamp(START_DATE)
        and weather["time"].max() == pd.Timestamp(END_DATE)
    )

    print("Rows received:", f"{len(weather):,}")
    print("Expected rows:", f"{expected_rows:,}")
    print("Missing municipality codes:", missing_codes)
    print("Duplicate municipality-date keys:", duplicate_keys)
    print("Complete date coverage:", complete_coverage)

    if not valid:
        raise ValueError(
            f"Batch {batch_id} failed validation. "
            "No output files were saved."
        )

    # Save temporary files first, then rename after successful writes.
    weather_tmp = weather_file.with_suffix(".csv.tmp")
    metadata_tmp = metadata_file.with_suffix(".csv.tmp")

    try:
        weather.to_csv(weather_tmp, index=False)
        metadata.to_csv(metadata_tmp, index=False)

        weather_tmp.replace(weather_file)
        metadata_tmp.replace(metadata_file)

    finally:
        if weather_tmp.exists():
            weather_tmp.unlink()
        if metadata_tmp.exists():
            metadata_tmp.unlink()

    # Confirm that saved files pass the same checks.
    if not check_batch(batch_id, batch_locations):
        raise ValueError(
            f"Batch {batch_id} failed validation after saving."
        )

    successful_this_run += 1
    print(f"Batch {batch_id} saved and validated successfully.")

    if position < len(pending_batches):
        print(
            f"Pausing {BATCH_PAUSE_SECONDS} seconds "
            "before the next batch."
        )
        time.sleep(BATCH_PAUSE_SECONDS)


# ---------------------------------------------------------
# Final status
# ---------------------------------------------------------

print("\n========== EXTRACTION SUMMARY ==========")
print("Newly validated batches:", successful_this_run)
print("Daily quota exceeded:", quota_exceeded)

final_validated = []
final_pending = []

for batch_id in batch_ids:
    locations = municipality_locations.loc[
        municipality_locations["batch_id"] == batch_id
    ]

    if check_batch(batch_id, locations):
        final_validated.append(batch_id)
    else:
        final_pending.append(batch_id)

print("All validated batches:", final_validated)
print("Remaining pending batches:", final_pending)
print("Remaining batch count:", len(final_pending))

remaining_rows = sum(
    len(
        municipality_locations.loc[
            municipality_locations["batch_id"] == batch_id
        ]
    ) * EXPECTED_DAYS
    for batch_id in final_pending
)

print("Remaining expected rows:", f"{remaining_rows:,}")

if quota_exceeded:
    print(
        "\nRun this same cell again after the API quota resets. "
        "Existing validated batches will be skipped."
    )

In [10]:


from pathlib import Path
import pandas as pd

# ---------------------------------------------------------
# 1. Locate project root
# ---------------------------------------------------------

current_dir = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        path
        for path in [current_dir, *current_dir.parents]
        if (path / "data" / "source").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"Could not locate the project root from: {current_dir}"
    )

EVA_PATH = (
    PROJECT_ROOT
    / "data"
    / "source"
    / "Evaluaciones_Agropecuarias_Municipales_–_EVA._2019_-_2025._Base_Agrícola_20260905.csv"
)

DIVIPOLA_PATH = PROJECT_ROOT / "data" / "source" / "divipola.csv"

# Verify paths before loading
print(f"Project root: {PROJECT_ROOT}")
print(f"EVA file exists: {EVA_PATH.exists()}")
print(f"DIVIPOLA file exists: {DIVIPOLA_PATH.exists()}")

if not EVA_PATH.is_file():
    raise FileNotFoundError(
        f"EVA file not found at: {EVA_PATH}\n"
        "Check the exact filename inside data/source."
    )

if not DIVIPOLA_PATH.is_file():
    raise FileNotFoundError(
        f"DIVIPOLA file not found at: {DIVIPOLA_PATH}"
    )

# ---------------------------------------------------------
# 2. Load source datasets
# ---------------------------------------------------------

eva = pd.read_csv(
    EVA_PATH,
    dtype="string",
    encoding="utf-8-sig",
)

divipola = pd.read_csv(
    DIVIPOLA_PATH,
    dtype="string",
    encoding="utf-8-sig",
)

print("\nDatasets loaded successfully.")
print(f"EVA rows: {len(eva):,}")
print(f"EVA columns: {len(eva.columns)}")
print(f"DIVIPOLA rows: {len(divipola):,}")
print(f"DIVIPOLA columns: {len(divipola.columns)}")





# ---------------------------------------------------------
# 3. Define source column names
# ---------------------------------------------------------

EVA_CODE = "Código Dane municipio"
EVA_DEPT_CODE = "Código Dane departamento"
EVA_DEPT = "Departamento"
EVA_MUNICIPALITY = "Municipio"

DIV_CODE = "Código Municipio"
DIV_DEPT_CODE = "Código Departamento"
DIV_DEPT = "Nombre Departamento"
DIV_MUNICIPALITY = "Nombre Municipio"
DIV_TYPE = "Tipo: Municipio / Isla / Área no municipalizada"


# ---------------------------------------------------------
# 4. Validate required columns
# ---------------------------------------------------------

required_eva = [
    EVA_CODE,
    EVA_DEPT_CODE,
    EVA_DEPT,
    EVA_MUNICIPALITY,
]

required_divipola = [
    DIV_CODE,
    DIV_DEPT_CODE,
    DIV_DEPT,
    DIV_MUNICIPALITY,
    DIV_TYPE,
    "Latitud",
    "longitud",
]

missing_eva = [c for c in required_eva if c not in eva.columns]
missing_divipola = [
    c for c in required_divipola if c not in divipola.columns
]

if missing_eva:
    raise ValueError(f"Missing EVA columns: {missing_eva}")

if missing_divipola:
    raise ValueError(f"Missing DIVIPOLA columns: {missing_divipola}")


# ---------------------------------------------------------
# 5. Normalize municipality codes
# ---------------------------------------------------------

def normalize_code(series):
    codes = series.astype("string").str.strip()
    codes = codes.str.replace(r"\.0$", "", regex=True)
    codes = codes.mask(codes.eq(""))
    return codes.str.zfill(5)


eva_geo = eva[required_eva].copy()

div_geo = divipola[required_divipola].copy()

eva_geo["municipality_code"] = normalize_code(eva_geo[EVA_CODE])
div_geo["municipality_code"] = normalize_code(div_geo[DIV_CODE])


# ---------------------------------------------------------
# 6. Identify malformed codes
# ---------------------------------------------------------

eva_bad_codes = eva_geo[
    eva_geo["municipality_code"].notna()
    & ~eva_geo["municipality_code"].str.fullmatch(r"\d{5}", na=False)
]

div_bad_codes = div_geo[
    div_geo["municipality_code"].notna()
    & ~div_geo["municipality_code"].str.fullmatch(r"\d{5}", na=False)
]

print("\n--- Code quality ---")
print(f"EVA rows with malformed codes: {len(eva_bad_codes):,}")
print(f"DIVIPOLA rows with malformed codes: {len(div_bad_codes):,}")

if not eva_bad_codes.empty:
    display(eva_bad_codes.head(20))

if not div_bad_codes.empty:
    display(div_bad_codes.head(20))


# ---------------------------------------------------------
# 7. Check whether EVA codes have conflicting attributes
# ---------------------------------------------------------

eva_unique = (
    eva_geo
    .dropna(subset=["municipality_code"])
    .drop_duplicates()
)

eva_conflicts = (
    eva_unique
    .groupby("municipality_code")
    .agg(
        municipality_names=(EVA_MUNICIPALITY, "nunique"),
        department_codes=(EVA_DEPT_CODE, "nunique"),
        department_names=(EVA_DEPT, "nunique"),
    )
    .query(
        "municipality_names > 1 "
        "or department_codes > 1 "
        "or department_names > 1"
    )
)

print("\n--- EVA attribute consistency ---")
print(f"Unique EVA municipality codes: {eva_unique['municipality_code'].nunique():,}")
print(f"Codes with conflicting attributes: {len(eva_conflicts):,}")

if not eva_conflicts.empty:
    display(eva_conflicts)


# ---------------------------------------------------------
# 8. Check duplicate codes in DIVIPOLA
# ---------------------------------------------------------

div_valid = div_geo.dropna(subset=["municipality_code"]).copy()

div_duplicate_codes = (
    div_valid[
        div_valid["municipality_code"].duplicated(keep=False)
    ]
    .sort_values("municipality_code")
)

print("\n--- DIVIPOLA code uniqueness ---")
print(f"Unique DIVIPOLA municipality codes: {div_valid['municipality_code'].nunique():,}")
print(f"Rows involved in duplicate codes: {len(div_duplicate_codes):,}")
print(
    "Distinct duplicated codes: "
    f"{div_duplicate_codes['municipality_code'].nunique():,}"
)

if not div_duplicate_codes.empty:
    display(div_duplicate_codes)


# ---------------------------------------------------------
# 9. Compare unique codes
# ---------------------------------------------------------

eva_code_set = set(eva_unique["municipality_code"].dropna().unique())
div_code_set = set(div_valid["municipality_code"].dropna().unique())

codes_in_both = eva_code_set & div_code_set
codes_only_eva = eva_code_set - div_code_set
codes_only_divipola = div_code_set - eva_code_set

print("\n--- Geographic code coverage ---")
print(f"Unique codes in EVA: {len(eva_code_set):,}")
print(f"Unique codes in DIVIPOLA: {len(div_code_set):,}")
print(f"Codes present in both sources: {len(codes_in_both):,}")
print(f"Codes only in EVA: {len(codes_only_eva):,}")
print(f"Codes only in DIVIPOLA: {len(codes_only_divipola):,}")


# ---------------------------------------------------------
# 10. Review EVA codes that do not match DIVIPOLA
# ---------------------------------------------------------

eva_unmatched = (
    eva_unique[
        eva_unique["municipality_code"].isin(codes_only_eva)
    ]
    .sort_values("municipality_code")
)

print("\n--- EVA municipalities without a DIVIPOLA match ---")
display(eva_unmatched)


# ---------------------------------------------------------
# 11. Check coordinate availability (not geographic accuracy)
# ---------------------------------------------------------

div_valid["latitude"] = pd.to_numeric(
    div_valid["Latitud"].str.replace(",", ".", regex=False),
    errors="coerce",
)

div_valid["longitude"] = pd.to_numeric(
    div_valid["longitud"].str.replace(",", ".", regex=False),
    errors="coerce",
)

missing_coordinates = div_valid[
    div_valid["latitude"].isna()
    | div_valid["longitude"].isna()
]

print("\n--- Coordinate availability ---")
print(f"DIVIPOLA rows with missing/unparseable coordinates: {len(missing_coordinates):,}")

if not missing_coordinates.empty:
    display(
        missing_coordinates[
            [
                "municipality_code",
                DIV_DEPT,
                DIV_MUNICIPALITY,
                DIV_TYPE,
                "Latitud",
                "longitud",
            ]
        ].head(30)
    )


# ---------------------------------------------------------
# 12. Check administrative categories for matched codes
# ---------------------------------------------------------

matched_divipola = div_valid[
    div_valid["municipality_code"].isin(codes_in_both)
]

print("\n--- Administrative categories for matched codes ---")
display(
    matched_divipola[DIV_TYPE]
    .value_counts(dropna=False)
    .rename_axis("Administrative type")
    .reset_index(name="Rows")
)


Project root: C:\Users\Monica\Desktop\ETL\ODS2
EVA file exists: True
DIVIPOLA file exists: True

Datasets loaded successfully.
EVA rows: 166,732
EVA columns: 18
DIVIPOLA rows: 1,122
DIVIPOLA columns: 7

--- Code quality ---
EVA rows with malformed codes: 0
DIVIPOLA rows with malformed codes: 0

--- EVA attribute consistency ---
Unique EVA municipality codes: 1,103
Codes with conflicting attributes: 0

--- DIVIPOLA code uniqueness ---
Unique DIVIPOLA municipality codes: 1,122
Rows involved in duplicate codes: 0
Distinct duplicated codes: 0

--- Geographic code coverage ---
Unique codes in EVA: 1,103
Unique codes in DIVIPOLA: 1,122
Codes present in both sources: 1,103
Codes only in EVA: 0
Codes only in DIVIPOLA: 19

--- EVA municipalities without a DIVIPOLA match ---


,Código Dane municipio,Código Dane departamento,Departamento,Municipio,municipality_code



--- Coordinate availability ---
DIVIPOLA rows with missing/unparseable coordinates: 0

--- Administrative categories for matched codes ---


,Administrative type,Rows
0,Municipio,1102
1,Isla,1


In [11]:

# ---------------------------------------------------------
# 1. Identify non-standard administrative categories
# ---------------------------------------------------------

matched_divipola = div_valid[
    div_valid["municipality_code"].isin(codes_in_both)
].copy()

non_municipal = matched_divipola[
    matched_divipola[DIV_TYPE].ne("Municipio")
]

print("Matched codes outside the standard Municipality category:")
display(
    non_municipal[
        [
            "municipality_code",
            DIV_DEPT,
            DIV_MUNICIPALITY,
            DIV_TYPE,
            "Latitud",
            "longitud",
        ]
    ]
)


# ---------------------------------------------------------
# 2. Build a normalized EVA municipality reference
# ---------------------------------------------------------

eva_municipalities = (
    eva_geo[
        [
            EVA_DEPT_CODE,
            EVA_DEPT,
            EVA_CODE,
            EVA_MUNICIPALITY,
            "municipality_code",
        ]
    ]
    .dropna(subset=["municipality_code"])
    .drop_duplicates(subset=["municipality_code"])
    .copy()
)

divipola_reference = div_valid[
    [
        "municipality_code",
        DIV_DEPT_CODE,
        DIV_DEPT,
        DIV_MUNICIPALITY,
        DIV_TYPE,
        "latitude",
        "longitude",
    ]
].copy()

# Use DIVIPOLA as the official source of coordinates.
municipality_reference = eva_municipalities.merge(
    divipola_reference,
    on="municipality_code",
    how="left",
    validate="one_to_one",
    indicator=True,
)

# ---------------------------------------------------------
# 3. Validate the resulting reference
# ---------------------------------------------------------

print("\n--- Municipality reference validation ---")
print(f"Reference rows: {len(municipality_reference):,}")
print(
    "Unique municipality codes: "
    f"{municipality_reference['municipality_code'].nunique():,}"
)
print(
    "Unmatched EVA codes: "
    f"{municipality_reference['_merge'].ne('both').sum():,}"
)
print(
    "Rows with missing coordinates: "
    f"{municipality_reference[['latitude', 'longitude']].isna().any(axis=1).sum():,}"
)
print("\nAdministrative categories:")
display(
    municipality_reference[DIV_TYPE]
    .value_counts(dropna=False)
    .rename_axis("Administrative type")
    .reset_index(name="Rows")
)

# ---------------------------------------------------------
# 4. Create the weather extraction catalog
# ---------------------------------------------------------

municipality_locations = (
    municipality_reference[
        [
            "municipality_code",
            EVA_DEPT_CODE,
            EVA_DEPT,
            EVA_MUNICIPALITY,
            "latitude",
            "longitude",
            DIV_TYPE,
        ]
    ]
    .sort_values("municipality_code")
    .reset_index(drop=True)
    .copy()
)

print("\nWeather extraction catalog created.")
display(municipality_locations.head())


Matched codes outside the standard Municipality category:


,municipality_code,Nombre Departamento,Nombre Municipio,Tipo: Municipio / Isla / Área no municipalizada,Latitud,longitud
1087,88001,"ARCHIPIÉLAGO DE SAN ANDRÉS, PROVIDENCIA Y SANT...",SAN ANDRÉS,Isla,"12,578108","-81,707181"



--- Municipality reference validation ---
Reference rows: 1,103
Unique municipality codes: 1,103
Unmatched EVA codes: 0
Rows with missing coordinates: 0

Administrative categories:


,Administrative type,Rows
0,Municipio,1102
1,Isla,1



Weather extraction catalog created.


,municipality_code,Código Dane departamento,Departamento,Municipio,latitude,longitude,Tipo: Municipio / Isla / Área no municipalizada
0,05001,05,Antioquia,Medellín,6.246631,-75.581775,Municipio
1,05002,05,Antioquia,Abejorral,5.789315,-75.428739,Municipio
2,05004,05,Antioquia,Abriaquí,6.632282,-76.064304,Municipio
3,05021,05,Antioquia,Alejandría,6.376061,-75.141346,Municipio
4,05030,05,Antioquia,Amagá,6.038708,-75.702188,Municipio


In [12]:

# ---------------------------------------------------------
# 1. Current catalog and expected extraction volume
# ---------------------------------------------------------

START_DATE = "2019-01-01"
END_DATE = "2025-12-31"
BATCH_SIZE = 25

expected_days = len(
    pd.date_range(start=START_DATE, end=END_DATE, freq="D")
)

municipality_locations = (
    municipality_locations
    .sort_values("municipality_code")
    .reset_index(drop=True)
    .copy()
)

municipality_locations["batch_id"] = (
    municipality_locations.index // BATCH_SIZE + 1
)

batch_plan = (
    municipality_locations
    .groupby("batch_id")
    .agg(
        municipalities=("municipality_code", "size"),
        first_code=("municipality_code", "first"),
        last_code=("municipality_code", "last"),
        first_municipality=("Municipio", "first"),
        last_municipality=("Municipio", "last"),
    )
    .reset_index()
)

expected_daily_records = (
    len(municipality_locations) * expected_days
)

print(f"Municipalities/territories: {len(municipality_locations):,}")
print(f"Days per territory: {expected_days:,}")
print(f"Expected daily records: {expected_daily_records:,}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Expected number of batches: {len(batch_plan):,}")

print("\n--- First five batches ---")
display(batch_plan.head())

print("\n--- Last three batches ---")
display(batch_plan.tail(3))


# ---------------------------------------------------------
# 2. Inspect existing batch files without modifying them
# ---------------------------------------------------------

WEATHER_OUTPUT_DIR = PROJECT_ROOT / "data" / "raw" / "open_meteo"

print(f"\nWeather output directory: {WEATHER_OUTPUT_DIR}")
print(f"Directory exists: {WEATHER_OUTPUT_DIR.is_dir()}")

if WEATHER_OUTPUT_DIR.is_dir():
    existing_weather_files = sorted(
        WEATHER_OUTPUT_DIR.glob("weather_batch_*.csv")
    )
    existing_metadata_files = sorted(
        WEATHER_OUTPUT_DIR.glob("location_metadata_batch_*.csv")
    )

    print(
        f"Existing weather batch files: "
        f"{len(existing_weather_files):,}"
    )
    print(
        f"Existing metadata batch files: "
        f"{len(existing_metadata_files):,}"
    )

    print("\nWeather files:")
    for path in existing_weather_files[:10]:
        print(f"  {path.name} ({path.stat().st_size:,} bytes)")
else:
    print(
        "Check WEATHER_OUTPUT_DIR against the output directory "
        "used by the previous extraction cell."
    )


Municipalities/territories: 1,103
Days per territory: 2,557
Expected daily records: 2,820,371
Batch size: 25
Expected number of batches: 45

--- First five batches ---


,batch_id,municipalities,first_code,last_code,first_municipality,last_municipality
0,1,25,05001,05120,Medellín,Cáceres
1,2,25,05125,05306,Caicedo,Giraldo
2,3,25,05308,05495,Girardota,Nechí
3,4,25,05501,05674,Olaya,San Vicente Ferrer
4,5,25,05679,05895,Santa Bárbara,Zaragoza



--- Last three batches ---


,batch_id,municipalities,first_code,last_code,first_municipality,last_municipality
42,43,25,81300,86219,Fortul,Colón
43,44,25,86320,99001,Orito,Puerto Carreño
44,45,3,99524,99773,La Primavera,Cumaribo



Weather output directory: C:\Users\Monica\Desktop\ETL\ODS2\data\raw\open_meteo
Directory exists: False
Check WEATHER_OUTPUT_DIR against the output directory used by the previous extraction cell.


In [13]:

from pathlib import Path
import pandas as pd

# ---------------------------------------------------------
# 1. Paths and validation parameters
# ---------------------------------------------------------

BATCH_DIR = PROJECT_ROOT / "data" / "raw" / "open_meteo_daily_batches"

START_DATE = pd.Timestamp("2019-01-01")
END_DATE = pd.Timestamp("2025-12-31")
EXPECTED_DAYS = len(pd.date_range(START_DATE, END_DATE, freq="D"))

print(f"Batch directory: {BATCH_DIR}")
print(f"Directory exists: {BATCH_DIR.is_dir()}")

if not BATCH_DIR.is_dir():
    raise FileNotFoundError(f"Batch directory not found: {BATCH_DIR}")

# Use the existing catalog and its established batch assignment.
catalog = municipality_locations.copy()

if "batch_id" not in catalog.columns:
    catalog = catalog.sort_values("municipality_code").reset_index(drop=True)
    catalog["batch_id"] = catalog.index // 25 + 1

catalog["municipality_code"] = (
    catalog["municipality_code"].astype("string").str.strip().str.zfill(5)
)

expected_dates = pd.date_range(START_DATE, END_DATE, freq="D")
results = []

# ---------------------------------------------------------
# 2. Validate each existing batch
# ---------------------------------------------------------

for batch_id in range(1, 4):
    path = BATCH_DIR / f"weather_batch_{batch_id:03d}.csv"

    if not path.is_file():
        print(f"\nMISSING: {path.name}")
        continue

    df = pd.read_csv(path, dtype={"municipality_code": "string"})

    # Normalize identifiers and dates.
    df["municipality_code"] = (
        df["municipality_code"].astype("string").str.strip().str.zfill(5)
    )
    df["date"] = pd.to_datetime(df["date"], errors="coerce")

    expected_codes = set(
        catalog.loc[catalog["batch_id"].eq(batch_id), "municipality_code"]
    )
    actual_codes = set(df["municipality_code"].dropna().unique())

    # Verify the actual municipality/date combinations.
    duplicate_pairs = int(
        df.duplicated(["municipality_code", "date"]).sum()
    )

    per_municipality = df.groupby("municipality_code")["date"].agg(
        records="size",
        distinct_dates="nunique",
        first_date="min",
        last_date="max",
    )

    complete_series = (
        len(actual_codes) == len(expected_codes)
        and actual_codes == expected_codes
        and len(per_municipality) == len(expected_codes)
        and per_municipality["records"].eq(EXPECTED_DAYS).all()
        and per_municipality["distinct_dates"].eq(EXPECTED_DAYS).all()
        and per_municipality["first_date"].eq(START_DATE).all()
        and per_municipality["last_date"].eq(END_DATE).all()
        and duplicate_pairs == 0
    )

    missing_codes = sorted(expected_codes - actual_codes)
    unexpected_codes = sorted(actual_codes - expected_codes)

    print(f"\n--- Batch {batch_id:03d} ---")
    print(f"File: {path.name}")
    print(f"Rows: {len(df):,}")
    print(f"Expected municipalities: {len(expected_codes)}")
    print(f"Actual municipalities: {len(actual_codes)}")
    print(f"Expected rows: {len(expected_codes) * EXPECTED_DAYS:,}")
    print(f"Duplicate municipality-date pairs: {duplicate_pairs}")
    print(f"Missing municipalities: {missing_codes}")
    print(f"Unexpected municipalities: {unexpected_codes}")
    print(f"Complete date series: {complete_series}")

    results.append({
        "batch_id": batch_id,
        "file": path.name,
        "rows": len(df),
        "expected_rows": len(expected_codes) * EXPECTED_DAYS,
        "municipalities": len(actual_codes),
        "expected_municipalities": len(expected_codes),
        "duplicate_pairs": duplicate_pairs,
        "missing_codes": len(missing_codes),
        "unexpected_codes": len(unexpected_codes),
        "complete": complete_series,
    })

validation_results = pd.DataFrame(results)

print("\n--- Validation summary ---")
display(validation_results)


Batch directory: C:\Users\Monica\Desktop\ETL\ODS2\data\raw\open_meteo_daily_batches
Directory exists: True


KeyError: 'municipality_code'

In [15]:

from pathlib import Path
import pandas as pd

BATCH_DIR = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "open_meteo_daily_batches"
)

catalog_check = load_municipality_catalog()

for batch_id in range(1, 4):
    path = BATCH_DIR / f"weather_batch_{batch_id:03d}.csv"

    existing = pd.read_csv(
        path,
        dtype={"municipality_code": "string"},
        usecols=["municipality_code", "latitude", "longitude"],
    )

    existing["municipality_code"] = normalize_code(
        existing["municipality_code"]
    )

    actual = existing.drop_duplicates(
        "municipality_code"
    ).sort_values("municipality_code")

    expected = catalog_check.loc[
        catalog_check["batch_id"].eq(batch_id),
        ["municipality_code", "latitude", "longitude"],
    ].sort_values("municipality_code")

    codes_match = (
        actual["municipality_code"].tolist()
        == expected["municipality_code"].tolist()
    )

    merged = actual.merge(
        expected,
        on="municipality_code",
        how="outer",
        suffixes=("_actual", "_expected"),
        indicator=True,
    )

    coordinates_match = (
        merged["_merge"].eq("both").all()
        and merged["latitude_actual"].notna().all()
        and merged["longitude_actual"].notna().all()
        and merged["latitude_expected"].notna().all()
        and merged["longitude_expected"].notna().all()
        and (
            merged["latitude_actual"]
            .sub(merged["latitude_expected"])
            .abs()
            .lt(1e-5)
            .all()
        )
        and (
            merged["longitude_actual"]
            .sub(merged["longitude_expected"])
            .abs()
            .lt(1e-5)
            .all()
        )
    )

    print(
        f"Batch {batch_id:03d}: "
        f"codes_match={codes_match}, "
        f"coordinates_match={coordinates_match}"
    )


NameError: name 'load_municipality_catalog' is not defined

In [16]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path(r"C:\Users\Monica\Desktop\ETL\ODS2")

source_dir = PROJECT_ROOT / "data" / "source"

divipola = pd.read_csv(
    source_dir / "divipola.csv",
    dtype="string",
)

eva = pd.read_csv(
    source_dir
    / "Evaluaciones_Agropecuarias_Municipales_–_EVA._2019_-_2025._Base_Agrícola_20260905.csv",
    dtype="string",
    low_memory=False,
)

def normalize_code(series):
    result = (
        series.astype("string")
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )
    result = result.mask(
        result.isna()
        | result.eq("")
        | result.str.lower().eq("nan")
    )
    return result.str.zfill(5)

eva_codes = set(
    normalize_code(eva["Código Dane municipio"])
    .dropna()
    .unique()
)

divipola["municipality_code"] = normalize_code(
    divipola["Código Municipio"]
)

divipola["latitude_numeric"] = pd.to_numeric(
    divipola["Latitud"],
    errors="coerce",
)

divipola["longitude_numeric"] = pd.to_numeric(
    divipola["longitud"],
    errors="coerce",
)

catalog = divipola.loc[
    divipola["municipality_code"].isin(eva_codes)
].copy()

invalid = catalog.loc[
    catalog["latitude_numeric"].isna()
    | catalog["longitude_numeric"].isna()
    | ~catalog["latitude_numeric"].between(-90, 90)
    | ~catalog["longitude_numeric"].between(-180, 180),
    [
        "municipality_code",
        "Nombre Departamento",
        "Nombre Municipio",
        "Latitud",
        "longitud",
        "latitude_numeric",
        "longitude_numeric",
    ],
]

print("Territories in EVA:", len(eva_codes))
print("Territories in catalog:", len(catalog))
print("Territories with invalid coordinates:", len(invalid))

display(invalid)

Territories in EVA: 1103
Territories in catalog: 1103
Territories with invalid coordinates: 1103


,municipality_code,Nombre Departamento,Nombre Municipio,Latitud,longitud,latitude_numeric,longitude_numeric
0,05001,ANTIOQUIA,MEDELLÍN,"6,246631","-75,581775",<NA>,<NA>
1,05002,ANTIOQUIA,ABEJORRAL,"5,789315","-75,428739",<NA>,<NA>
2,05004,ANTIOQUIA,ABRIAQUÍ,"6,632282","-76,064304",<NA>,<NA>
3,05021,ANTIOQUIA,ALEJANDRÍA,"6,376061","-75,141346",<NA>,<NA>
4,05030,ANTIOQUIA,AMAGÁ,"6,038708","-75,702188",<NA>,<NA>
...,...,...,...,...,...,...,...
1115,97666,VAUPÉS,TARAIRA,"-0,564984","-69,635497",<NA>,<NA>
1118,99001,VICHADA,PUERTO CARREÑO,"6,186636","-67,487095",<NA>,<NA>
1119,99524,VICHADA,LA PRIMAVERA,"5,486309","-70,410515",<NA>,<NA>
1120,99624,VICHADA,SANTA ROSALÍA,"5,136393","-70,859499",<NA>,<NA>
